<a href="https://colab.research.google.com/github/Ram-Vidhu/Job_searcher_and_Resume_Enhancer/blob/users%2Fvidhya%2Fgen_ai/Notebooks/GenAI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Spark Setup

In [1]:
! pip install pyspark

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Job_recommendation").getOrCreate()

## Data preprocessing

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
df1 = spark.read.format('csv').option("header", True).load('..\Datasets\job_skills.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\vidhy\AppData\Local\Temp\ipykernel_9256\2020187667.py:1: SyntaxWarning: invalid escape sequence '\D'
  df1 = spark.read.format('csv').option("header", True).load('..\Datasets\job_skills.csv')


In [6]:
df2 = spark.read.format('csv').option("header", True).load('..\Datasets\job_summary.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\vidhy\AppData\Local\Temp\ipykernel_9256\1740858380.py:1: SyntaxWarning: invalid escape sequence '\D'
  df2 = spark.read.format('csv').option("header", True).load('..\Datasets\job_summary.csv')


In [7]:
df3 = df1.join(df2,on='job_link',how='inner')

In [8]:
df4 = spark.read.format('csv').option("header", True).load('..\Datasets\linkedin_job_postings.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\vidhy\AppData\Local\Temp\ipykernel_9256\826778073.py:1: SyntaxWarning: invalid escape sequence '\D'
  df4 = spark.read.format('csv').option("header", True).load('..\Datasets\linkedin_job_postings.csv')


In [9]:
df5 = df3.join(df4,on='job_link',how='inner')
df5.drop('last_processed_time','got_summary','got_ner','is_being_worked')

DataFrame[job_link: string, job_skills: string, job_summary: string, job_title: string, company: string, job_location: string, first_seen: string, search_city: string, search_country: string, search_position: string, job_level: string, job_type: string]

## EDA

In [11]:
from pyspark.sql import functions as f

# take null counts
null_counts = df5.select([
    f.sum(f.col(c).isNull().cast("int")).alias(c)
    for c in df5.columns
])

null_counts.show()


+--------+----------+-----------+-------------------+-----------+-------+---------------+---------+-------+------------+----------+-----------+--------------+---------------+---------+--------+
|job_link|job_skills|job_summary|last_processed_time|got_summary|got_ner|is_being_worked|job_title|company|job_location|first_seen|search_city|search_country|search_position|job_level|job_type|
+--------+----------+-----------+-------------------+-----------+-------+---------------+---------+-------+------------+----------+-----------+--------------+---------------+---------+--------+
|       0|      2007|          0|                  0|          0|      0|              0|        0|     40|          50|        31|         31|            31|             31|       31|      32|
+--------+----------+-----------+-------------------+-----------+-------+---------------+---------+-------+------------+----------+-----------+--------------+---------------+---------+--------+



In [13]:
# dropping null values
df5 = df5.na.drop()

## Storing and querying vectordb

In [14]:
!pip install chromadb

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached click-8.2.1-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   -- ------------------------------------- 1.3/19.8 MB 9.5 MB/s eta 0:00:02
   ------ --------------------------------- 3.4/19.8 MB 10.1 MB/s eta 0:00:02
   ----------- ---------------------------- 5.8/19.8 MB 10.5 MB/s eta 0:00:02
   -------------- ------------------------- 7.3/19.8 MB 9.3 MB/s eta 0:00:02
   ----------------- ---------------------- 8.7/19.8 MB 8.8 MB/s eta 0:00:02
   ------------------ --------------------- 9.2/19.8 MB 8.7 MB/s eta 0:00:02
   ------------------- -------------------- 9.4/19.8 MB 7.0 MB/s eta 0:00:

In [16]:
!pip install sentence_transformers

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ------- -------------------------------- 2.1/11.6 MB 10.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.9/11.6 MB 10.1 MB/s eta 0:00:01
   ------------------- -------------------- 5.8/11.6 MB 9.2 MB/s eta 0:00:01
   -------------------------- ------------- 7.6/11.6 MB 8.9 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.6 MB 8.7 MB/s eta 0:00:01
   ------------------------------------- -- 11.0/11.6 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 8.3 MB/s  0:00:01
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 1.6/241.3 MB 8.2 MB/s eta 0:00:30
    --------------------------------------- 3.4/241.3 MB 8.3 MB/s eta 0:00:29
    --------------------------------------- 5.2/241.3 MB 8.3 MB/s eta 0:00:29
   - -------------------------------------- 7.1/241.3 MB 8.3 MB/s eta 0:00:29
   - --

In [17]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [18]:
from pyspark.sql import functions as F

df5 = df5.withColumn(
    "job_text",
    F.concat_ws(
        " ",   # separator
        F.coalesce(F.col("job_title"), F.lit("")),
        F.coalesce(F.col("job_summary"), F.lit("")),
        F.coalesce(F.col("job_skills"), F.lit("")),
        F.coalesce(F.col("job_level"), F.lit(""))
    )
)

In [19]:
# Init Chroma client (persistent storage)
client = chromadb.PersistentClient(path="chroma_db")

# Create or get collection
collection = client.get_or_create_collection(
    name="jobs",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

In [20]:
# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\vidhy\miniconda3\envs\gen_ai\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vidhy\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
def clean_metadata(row_dict):
    clean = {}
    for k, v in row_dict.items():
        if k == "job_text":   # don't include job_text in metadata
            continue
        if v is None:
            clean[k] = ""   # default to empty string
        elif isinstance(v, (bool, int, float, str)):
            clean[k] = v
        else:
            clean[k] = str(v)   # fallback: convert to string
    return clean

In [ ]:
batch_size = 500
rows_iter = df5.toLocalIterator()

batch = []
for row in rows_iter:
    batch.append(row.asDict())

    if len(batch) >= batch_size:
        texts = [r["job_text"] for r in batch]
        embeddings = model.encode(texts)

        ids = [str(i) for i in range(len(batch))]
        metadata = [clean_metadata(r) for r in batch]

        collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadata
        )
        batch = []

In [ ]:
def search_jobs_chroma(resume_text, top_k=5, filters=None):
    embedding = model.encode([resume_text])[0]

    results = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=top_k,
        where=filters  # e.g., {"job_location": "Berlin", "job_type": "Full-time"}
    )

    jobs = []
    for i in range(len(results["ids"][0])):
        jobs.append({
            "similarity_score": results["distances"][0][i],
            **results["metadatas"][0][i]}
        )
        return pd.DataFrame(jobs)

In [ ]:
results = search_jobs_chroma("data scientist [python, sql, machine learning, pyspark, Azure] mid level", top_k=5)

In [ ]:
results